In [15]:
import torch
import torch.nn as nn

import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"PyTorch version : {torch.__version__}")
print(f"Running on : {device.upper()}")

## Load and Prepare Data
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print("\n--------Samples----------")
print(f"Train : {X_train.shape}  ({len(X_train)} samples)")
print(f"Test  : {X_test.shape}   ({len(X_test)} samples)")
print()

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"After scaling — Train mean : {X_train.mean(axis=0).round(2)}  (should be ≈ 0)")
print(f"After scaling — Train std  : {X_train.std(axis=0).round(2)}   (should be ≈ 1)")
print()

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_t = torch.tensor(X_test,  dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
y_test_t = torch.tensor(y_test,  dtype=torch.long).to(device)

PyTorch version : 2.10.0+cu128
Running on : CUDA

--------Samples----------
Train : (120, 4)  (120 samples)
Test  : (30, 4)   (30 samples)

After scaling — Train mean : [-0. -0.  0.  0.]  (should be ≈ 0)
After scaling — Train std  : [1. 1. 1. 1.]   (should be ≈ 1)



# Add Third Hidden Layer

In [5]:
class IrisNetwork(nn.Module):
  def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(4, 16)
        self.layer2 = nn.Linear(16,8)
        self.relu = nn.ReLU()
        self.layer3 = nn.Linear(8, 3)

  def forward(self, x):
      x = self.layer1(x)
      x = self.relu(x)
      x = self.layer2(x)
      x = self.relu(x)
      x = self.layer3(x)
      return x

model = IrisNetwork().to(device)
print(model)
print()

IrisNet(
  (layer1): Linear(in_features=4, out_features=16, bias=True)
  (layer2): Linear(in_features=16, out_features=8, bias=True)
  (relu): ReLU()
  (layer3): Linear(in_features=8, out_features=3, bias=True)
)



In [6]:
total = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {total}")

Total trainable parameters: 243


In [11]:
import math

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

model.eval()
with torch.no_grad():
    init_loss = loss_fn(model(X_train_t.to(device)), y_train_t.to(device))
print(f"Initial loss: {init_loss.item():.4f}   (expected ≈ {math.log(3):.4f})")
print()

Initial loss: 1.0960   (expected ≈ 1.0986)



In [17]:
train_losses = []
EPOCHS=200

for epoch in range(1, EPOCHS + 1):

    model.train()
    logits = model(X_train_t.to(device))
    loss = loss_fn(logits, y_train_t.to(device))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    if (epoch+1) % 40 == 0:
        model.eval()
        with torch.no_grad():
            correct = (model(X_test_t).argmax(1) == y_test_t).sum().item()
            acc = correct / len(y_test_t) * 100
        print(f"Epoch {epoch+1} | Loss: {loss.item():.4f} | Accuracy: {acc:.1f}%")

Epoch 40 | Loss: 0.0267 | Accuracy: 93.3%
Epoch 80 | Loss: 0.0262 | Accuracy: 93.3%
Epoch 120 | Loss: 0.0256 | Accuracy: 93.3%
Epoch 160 | Loss: 0.0250 | Accuracy: 93.3%
Epoch 200 | Loss: 0.0245 | Accuracy: 93.3%


# Removed ReLU

In [18]:
class IrisNet_NoReLU(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(4, 16)
        self.layer2 = nn.Linear(16,8)
        self.layer3 = nn.Linear(8, 3)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return x

model     = IrisNet_NoReLU().to(device)
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
    model.train()
    y_pred = model(X_train_t)
    loss = loss_fn(y_pred, y_train_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 40 == 0:
        model.eval()
        with torch.no_grad():
            correct = (model(X_test_t).argmax(1) == y_test_t).sum().item()
            acc = correct / len(y_test_t) * 100
        print(f"Epoch {epoch+1} | Loss: {loss.item():.4f} | Accuracy: {acc:.1f}%")

Epoch 40 | Loss: 0.1154 | Accuracy: 93.3%
Epoch 80 | Loss: 0.0458 | Accuracy: 96.7%
Epoch 120 | Loss: 0.0416 | Accuracy: 100.0%
Epoch 160 | Loss: 0.0405 | Accuracy: 100.0%
Epoch 200 | Loss: 0.0403 | Accuracy: 100.0%


# Remove Standard Scaler

In [20]:
iris = load_iris()
X_raw = iris.data
y_raw = iris.target

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_raw, y_raw, test_size=0.2, random_state=SEED)

X_train_r = torch.tensor(X_train_r, dtype=torch.float32).to(device)
X_test_r = torch.tensor(X_test_r,  dtype=torch.float32).to(device)
y_train_r = torch.tensor(y_train_r, dtype=torch.long).to(device)
y_test_r = torch.tensor(y_test_r,  dtype=torch.long).to(device)

class IrisNet_NoScale(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(4, 16)
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(16, 3)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

model = IrisNet_NoScale().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
    model.train()
    y_pred = model(X_train_r)
    loss = loss_fn(y_pred, y_train_r)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 40 == 0:
        model.eval()
        with torch.no_grad():
            correct = (model(X_test_r).argmax(1) == y_test_r).sum().item()
            acc = correct / len(y_test_r) * 100
        print(f"Epoch {epoch+1} | Loss: {loss.item():.4f} | Accuracy: {acc:.1f}%")

Epoch 40 | Loss: 0.2727 | Accuracy: 100.0%
Epoch 80 | Loss: 0.1038 | Accuracy: 100.0%
Epoch 120 | Loss: 0.0736 | Accuracy: 100.0%
Epoch 160 | Loss: 0.0637 | Accuracy: 100.0%
Epoch 200 | Loss: 0.0590 | Accuracy: 100.0%


# Wine Dataset

In [46]:
from sklearn.datasets import load_wine

wine = load_wine()
X_w = wine.data
y_w =  wine.target

print(X_w.shape)

X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(X_w, y_w, test_size=0.2, random_state=SEED)

scaler = StandardScaler()
X_train_w = scaler.fit_transform(X_train_w)
X_test_w = scaler.transform(X_test_w)

X_train_w = torch.tensor(X_train_w, dtype=torch.float32).to(device)
X_test_w = torch.tensor(X_test_w,  dtype=torch.float32).to(device)
y_train_w = torch.tensor(y_train_w, dtype=torch.long).to(device)
y_test_w = torch.tensor(y_test_w,  dtype=torch.long).to(device)

class WineNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(13, 16)
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(16, 3)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

model = WineNet().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
    model.train()
    y_pred = model(X_train_w)
    loss = loss_fn(y_pred, y_train_w)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 40 == 0:
        model.eval()
        with torch.no_grad():
            correct = (model(X_test_w).argmax(1) == y_test_w).sum().item()
            acc = correct / len(y_test_w) * 100
        print(f"Epoch {epoch+1} | Loss: {loss.item():.4f} | Accuracy: {acc:.1f}%")

(178, 13)
Epoch 40 | Loss: 0.0334 | Accuracy: 100.0%
Epoch 80 | Loss: 0.0090 | Accuracy: 100.0%
Epoch 120 | Loss: 0.0044 | Accuracy: 100.0%
Epoch 160 | Loss: 0.0026 | Accuracy: 100.0%
Epoch 200 | Loss: 0.0018 | Accuracy: 100.0%


# Mini Batching

In [47]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X_train_t, y_train_t)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

class IrisNet_Batch(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(4, 16)
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(16, 3)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

model = IrisNet_Batch().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
    model.train()
    for X_batch, y_batch in dataloader:   # loop over mini batches
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 40 == 0:
        model.eval()
        with torch.no_grad():
            correct = (model(X_test_t).argmax(1) == y_test_t).sum().item()
            acc = correct / len(y_test_t) * 100
        print(f"Epoch {epoch+1} | Loss: {loss.item():.4f} | Accuracy: {acc:.1f}%")

Epoch 40 | Loss: 0.0056 | Accuracy: 96.7%
Epoch 80 | Loss: 0.0004 | Accuracy: 96.7%
Epoch 120 | Loss: 0.0008 | Accuracy: 96.7%
Epoch 160 | Loss: 0.0231 | Accuracy: 96.7%
Epoch 200 | Loss: 0.0002 | Accuracy: 96.7%
